In [0]:
#************************************************************************************************************************************
#*                                                                                                                                  *
#*   NOTEBOOK:     Auto_Run_EDW_Data_Load.                                                                                       *
#*                                                                                                                                  *
#*   DESCRIPTION:                                                                                                                   *
#*                                                                                                                                  *
#*                                                                                                                                  *
#*   INPUT PARMS:                                                                                                                   *
#*                                                                                                                                  *
#*                                                                                                                                  *
#*   INPUT FILES:                                                                                                                   *
#*                                                                                                                                  *
#*                                                                                                                                  *
#*   OUTPUT FILE:                                                                                                                   *
#*                                                                                                                                  *
#*   EXITS:       0 - success                                                                                                       *
#*                <> 0 - failure                                                                                                    *
#*                                                                                                                                  *
#************************************************************************************************************************************
#*                                                                                                                                  *
#*                                                 Modification Log                                                                 *
#*                                                                                                                                  *
#*    Date     CO                 Author              Description                                                                   *
#* ---------- ------------------  -----------------   ------------------------------------------------------------------------------*
# * 25/06/2025 000000000000000001  Ahmad Afzaal        Hash file checking logic has been added to enhance the finding of .ctl file  *
# * 05/06/2025 000000000000000000  Ahmad Afzaal        Initial Release.                                                             *
#************************************************************************************************************************************


In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("base_path", "s3a://gia-stg-oh-ue1-data-raw/haven/inbound/VE_EDW/weekly")
dbutils.widgets.text("log_table", "oh_apm_stg.tmp.ctl_reconciliation_log_Prv")
dbutils.widgets.text("delta_table", "oh_apm_stg.vendor_extracts.load_report_log_cpc_Stg_Prv") #temp tables Staging 
dbutils.widgets.text("copy_target_path", "s3://gia-stg-oh-ue1-data-raw/haven/inbound/VE_EDW/process/ETL_Prv")
dbutils.widgets.text("ctl_file_identifier", "ODM.EDW.VEN.PROVIDER.WEEKLY")
dbutils.widgets.text("columns_to_check_for_nulls", "SAK_RECIP")  # comma-separated

In [0]:
base_path = dbutils.widgets.get("base_path")
log_table = dbutils.widgets.get("log_table")
delta_table = dbutils.widgets.get("delta_table")
copy_target_path = dbutils.widgets.get("copy_target_path")
ctl_file_identifier = dbutils.widgets.get("ctl_file_identifier")
columns_to_check_for_nulls = dbutils.widgets.get("columns_to_check_for_nulls").split(",")


In [0]:
import os
import json
import re
from datetime import datetime, timedelta, timezone
from pyspark.sql import functions as f
from pyspark.dbutils import DBUtils
from pyspark.sql.types import StructType, StructField, StringType, LongType
from pyspark.sql import SparkSession
from pyspark.sql import Row
from pyspark.sql.functions import col
import pyspark.sql.functions as sqlf
import hashlib
# Initialize Spark session
spark = SparkSession.builder.appName("CTL-GZ-Reconciliation").getOrCreate()

#Error load report flags

critical_error_flag = False
soft_error_flag = False

# Recursive file listing
def list_all_files(path):
    items = []
    for fi in dbutils.fs.ls(path):              
        if fi.isDir():
            items += list_all_files(fi.path)
        else:
            items.append(fi)
    return items

all_files = list_all_files(base_path)

top_folders = [d for d in dbutils.fs.ls(base_path) if d.isDir()]

folder_mods = []
for d in top_folders:
    try:
        # find the newest modificationTime among its children
        child_times = [
            datetime.fromtimestamp(f.modificationTime/1000, tz=timezone.utc)
            for f in dbutils.fs.ls(d.path)
        ]
        folder_mods.append((d.name, max(child_times)))
    except Exception:
        pass

# sort descending by that max timestamp
folder_mods.sort(key=lambda x: x[1], reverse=True)
last_folder_landed = folder_mods[0][0] if folder_mods else None
print(f"📂 Latest folder landed under S3: {last_folder_landed}")

# Compute hash for each CTL file
def compute_hash(path):
    try:
        # Read first 5MB for hash — enough to uniquely identify most CTLs
        data = dbutils.fs.head(path, 5 * 1024 * 1024)
        return hashlib.sha256(data.encode("utf-8")).hexdigest()
    except Exception as e:
        print(f"❌ Could not hash {path}: {e}")
        return None

ctl_file_objs = [
    {
        "name": fi.name,
        "path": fi.path,
        "mod_time": datetime.fromtimestamp(fi.modificationTime / 1000).replace(tzinfo=timezone.utc),
        "hash": compute_hash(fi.path)
    }
    for fi in all_files
    if fi.name.endswith(".ctl") and ctl_file_identifier in fi.name
]

if not ctl_file_objs:
    raise Exception("❌ No .ctl files found to process")

# Load reconciliation log
log_df = spark.read.table(log_table).select(
    "CTL_File", "Status", "Processed_Timestamp", "CTL_Hash"
)

success_hashes = set([
    r["CTL_Hash"] for r in log_df.filter(col("Status") == "SUCCESS").collect() if r["CTL_Hash"]
])

error_hashes = {
    r["CTL_Hash"]: r["CTL_File"]
    for r in log_df.filter(col("Status") != "SUCCESS").collect() if r["CTL_Hash"]
}
# Match based on ctl_file_identifier
log_success_df = log_df.filter(
    (col("Status") == "SUCCESS") & col("CTL_File").contains(ctl_file_identifier)
)

success_count = log_success_df.count()
print(f"🎯 Number of successful CTLs matching '{ctl_file_identifier}': {success_count}")

if success_count > 0:
    latest_success_ctl = log_success_df.orderBy(f.desc("Processed_Timestamp")).limit(1).collect()[0]
    ctl_file_path_fragment = latest_success_ctl["CTL_File"]  # Example: ODM.EDW.VEN.REFERENCE.WEEKLY.20250619.ctl

    # Extract folder name if CTL path has a recognizable date or dt pattern (this is optional logic)
    matched_ctl_file_path = next(
        (f.path for f in all_files if f.name == ctl_file_path_fragment),
        None
    )

    if matched_ctl_file_path:
        ctl_folder = os.path.dirname(matched_ctl_file_path)
        print(f"📁 Folder of last successful CTL: {ctl_folder}")
        print(f"📄 Last successful CTL: {ctl_file_path_fragment}")
else:
    print("ℹ️ No previously successful CTL in log.")
# Select CTL to process

# 5) Select CTL based on freshness + uniqueness

cutoff_date = datetime.now(timezone.utc) - timedelta(days=7)
existing_hashes = set(row["CTL_Hash"] for row in log_df.select("CTL_Hash").dropna().distinct().collect())

eligible_ctl_objs = []

for ctl in ctl_file_objs:
    path = ctl["path"]
    mod_time = ctl["mod_time"]

    if mod_time < cutoff_date:
        continue  # skip old files even if manually modified

    ctl_hash = compute_hash(path)
    if ctl_hash and ctl_hash not in existing_hashes:
        ctl["hash"] = ctl_hash
        eligible_ctl_objs.append(ctl)

# Sort newest first
eligible_ctl_objs.sort(key=lambda x: x["mod_time"], reverse=True)

if not eligible_ctl_objs:
    print("✅ No recent and unprocessed CTLs found in S3 Location.")
    raise Exception("No CTLs to process")

to_process = eligible_ctl_objs[0]
print(f"\n📄 CTL selected for processing: {to_process['name']}")
print(f"   📄 → Landed: {to_process['mod_time']:%Y-%m-%d %H:%M:%S}")

# 7) Read CTL and parse GZ expectations

gz_files_info = {
    fi.name: fi.path
    for fi in all_files
    if fi.name.endswith(".gz")
}

ctl_df = spark.read.text(to_process["path"])
ctl_data = ctl_df.collect()

gz_expected_info = []
gz_file_paths = {}

for row in ctl_data:
    parts = row[0].split('|')
    if len(parts) >= 2:
        file_name = parts[0].strip()
        expected_count = int(parts[1].strip())

        # Lookup full path in gz_files_info
        full_path = gz_files_info.get(file_name)

        if full_path:
            gz_expected_info.append((file_name, expected_count))
            gz_file_paths[file_name] = full_path  # ✅ store path separately
        else:
            raise Exception(f"❌ Expected GZ file listed in CTL but not found in any S3 folder: {file_name}")

# Step 4: Save gz_expected_info to taskValues for QA notebook
dbutils.jobs.taskValues.set(key="gz_expected_info", value=gz_expected_info)
dbutils.jobs.taskValues.set(key="gz_file_paths", value=gz_file_paths)


# 8) Build GZ lookup across all files

missing = [name for name, _ in gz_expected_info if name not in gz_files_info]
if missing:
    raise Exception(f"❌ Missing .gz files for this CTL: {missing}")

# Reconciliation
results = []
has_critical_failure = False
for gz_file_name, expected_count in gz_expected_info:
    matching_file_path = gz_files_info.get(gz_file_name)
    null_columns_failed = []
    error_count = None
    status = None
    actual_count = None

    if matching_file_path:
        try:
            # Check if file is non-empty
            if dbutils.fs.ls(matching_file_path):
                gz_df = spark.read.option("header", "true").option("inferSchema", "true").csv(matching_file_path)
                actual_count = gz_df.count()

                # Null check
                for col_name in columns_to_check_for_nulls:
                    if col_name in gz_df.columns:
                        null_count = gz_df.filter(col(col_name).isNull()).count()
                        if null_count > 0:
                            null_columns_failed.append(col_name)

                if null_columns_failed:
                    status = "NULL_VALUES_FOUND"
                    has_critical_failure = True
                    total_nulls = sum(gz_df.filter(col(col_name).isNull()).count() for col_name in null_columns_failed)
                    error_count = total_nulls
                elif actual_count != expected_count:
                    status = "Mismatch"
                    error_count = abs(actual_count - expected_count)
                else:
                    status = "SUCCESS"
                    error_count = 0
            else:
                status = "EMPTY_FILE"
                actual_count = 0
                has_critical_failure = True
        except Exception as e:
            status = "CORRUPT"
            actual_count = None
            has_critical_failure = True
    else:
        status = "MISSING"
        actual_count = None
        has_critical_failure = True

    results.append(Row(
        CTL_File=to_process["name"],
        GZ_File=gz_file_name,
        Status=status,
        Row_Count_in_CTL=expected_count,
        Row_Count_in_GZ=actual_count,
        Null_Columns_Failed=",".join(null_columns_failed) if null_columns_failed else None,
        Number_of_Error_Records=error_count
    ))

    gz_files = [r['GZ_File'] for r in results if r['Status'] not in ["NULL_VALUES_FOUND", "MISSING", "CORRUPT"]]
    dbutils.jobs.taskValues.set(key="gz_file_list", value=gz_files)

# Display and log results
if results:
    result_df = spark.createDataFrame(results, schema="CTL_File STRING, GZ_File STRING, Status STRING, Row_Count_in_CTL INT, Row_Count_in_GZ INT, Null_Columns_Failed STRING, Number_of_Error_Records bigint")
    result_df = result_df.withColumn("Processed_Timestamp", sqlf.current_timestamp())
    print("\nFinal Reconciliation Report:")
    result_df.show(truncate=False)

    # Append to Unity Catalog Delta table
    result_df = result_df.withColumn("CTL_Hash", f.lit(to_process["hash"]))  # Add hash to result
    result_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(log_table)

if has_critical_failure:
    #Eror load report
    critical_error_flag = True
    dbutils.jobs.taskValues.set(key="critical_error_flag", value=critical_error_flag)
    raise Exception("❌ One or more critical reconciliation failures detected. Stopping workflow.")

ctl_received_ts = to_process["mod_time"].strftime("%Y-%m-%d %H:%M:%S")
if results:
    schema = StructType([
        StructField("File_Name", StringType(), True),
        StructField("Date_Received_by_APM", StringType(), True),
        StructField("Status", StringType(), True),
        StructField("Number_of_Records_Received", LongType(), True),
        StructField("Number_of_Records_Loaded", LongType(), True),
        StructField("Number_of_Error_Records", LongType(), True),
        StructField("Start_Load_Date", StringType(), True),
        StructField("End_Load_Date", StringType(), True)
    ])

    data = [
        (
            r['GZ_File'],  # File_Name
            ctl_received_ts,  # Date_Received_by_APM
            r[2],  # Status
            r[3],  # Number_of_Records_Received
            r[4],  # Number_of_Records_Loaded
            r['Number_of_Error_Records'],
            None,  # Start_Load_Date
            None   # End_Load_Date
        )
        for r in results if r['Status'] not in ["NULL_VALUES_FOUND", "MISSING", "CORRUPT"]
    ]

if not has_critical_failure:
    if any(r['Status'] == "Mismatch" for r in results):
        print("⚠️ Non-critical failure detected. Writing 'Mismatch' status and stopping workflow.")
        # at this stage we are not loading this mismatch error to data load report for rangan.
        if data:
            df = spark.createDataFrame(data, schema=schema)
            df.write.format("delta").mode("overwrite").saveAsTable(delta_table)
            print("📄 Wrote 'Mismatch' status to Delta table.")
            #error load report flag
            soft_error_flag = True
            dbutils.jobs.taskValues.set(key="soft_error_flag", value=soft_error_flag)
        raise Exception("❌ Pre‐Validation encountered a non-critical failure. Stopping workflow.")
    else:
        # All statuses are success
        print("✅ No critical failure and all statuses are 'Success'. Proceeding with data load and file operations.")
        # at this stage when everything is success we will load to rangans tables
        if data:
            df = spark.createDataFrame(data, schema=schema)
            df.write.format("delta").mode("overwrite").saveAsTable(delta_table)
            # Flag to upload data into APM tables
            run_condition = True  # or False based on your logic
            dbutils.jobs.taskValues.set(key="run_flag", value=run_condition)
            print("📄 Uploaded to Delta table.")

        # Clean folder
        if not copy_target_path.endswith("/"):
            copy_target_path += "/"

        def folder_exists(path):
            try:
                _ = dbutils.fs.ls(path)
                return True
            except Exception:
                return False

        if folder_exists(copy_target_path):
            files_in_target = dbutils.fs.ls(copy_target_path)
            for f in files_in_target:
                try:
                    dbutils.fs.rm(f.path, recurse=True)
                except Exception as e:
                    print(f"❌ Failed to delete {f.path}: {e}")
            print(f"🧹 Cleared files inside {copy_target_path} (folder preserved)")
        else:
            print(f"📁 Folder {copy_target_path} does not exist, skipping deletion step")

        # Copy CTL
        try:
            dbutils.fs.cp(to_process["path"], copy_target_path + to_process["name"])
            print(f"📄 Copied CTL file: {to_process['name']}")
        except Exception as e:
            raise Exception(f"❌ Failed to copy CTL file: {e}")

        # Copy GZ files
        for gz_file_name, _ in gz_expected_info:
            gz_path = gz_files_info.get(gz_file_name)
            if gz_path:
                try:
                    dbutils.fs.cp(gz_path, copy_target_path + gz_file_name)
                    print(f"📦 Copied GZ file: {gz_file_name}")
                except Exception as e:
                    raise Exception(f"❌ Failed to copy GZ file: {gz_file_name}, error: {e}")

        # # Extract and return only the date portion from ctl_received_ts
            
            ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
            run_id = ctx.tags().get("runId")

            # # Convert to string before storing
            if run_id:
                dbutils.jobs.taskValues.set(key="run_id", value=str(run_id))
            else:
                raise ValueError("❌ run_id not found in context. This notebook must be run as part of a job.")

else:
    print("⛔ Critical failure detected. Skipping all operations.")